# 11 · From notebook to package

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 2 | about 25 minutes live | [10 · Testing with pytest](10_testing_with_pytest.ipynb) | [12 · Capstone](12_capstone.ipynb) |

A notebook is a great place to explore. It is a poor place to keep code that other code depends on: you cannot
import a cell, you cannot test it from the terminal, a scheduler cannot run it reliably, and a reviewer cannot
see a clean diff. In this notebook you see how this repository turns the churn project into an installable
Python **package**, and how the pieces you met over the last two days (classes, config, errors, logging,
tests) fit into one project layout.

> **Learning goals.** By the end you can
>
> - explain why reusable code belongs in a package and not in a notebook
> - find your way around a `src` layout project and say what each top-level folder is for
> - import from an installed package and run it from the command line with `python -m`
> - read the main sections of a `pyproject.toml`
> - set up the structure of your own package for the capstone

**Before you start.** Notebooks 07 to 10. You need a terminal open in the repository root.

Exercises come in three levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for
experienced programmers).

**In this notebook**

1. Why notebooks are not enough
2. A tour of this repository
3. What `uv sync` did for you
4. Modules, scripts and `if __name__ == "__main__":`
5. `pyproject.toml`: one file for the whole project
6. Quality tools you run from the terminal
7. Your package for the capstone

---

## 1. Why notebooks are not enough

Think back to the warm-up notebook (00). It worked. But imagine these requests from your team:

| Request | What goes wrong with a notebook |
|---|---|
| "Can I reuse your cleaning code in my notebook?" | They copy cells. Now there are two versions that drift apart. |
| "Can we test the encoder?" | pytest runs files, not cells. |
| "Run it every night at 2am." | Cells depend on execution order and hidden state. |
| "Please review your change." | A notebook diff is JSON plus outputs, hard to read. |
| "It works on my machine..." | Paths and imports depend on where the notebook was started. |

The answer is not "never use notebooks". It is a division of labour:

- **Notebooks** explore, explain and visualise. They *import* the project code.
- **The package** (`src/...`) holds every piece of logic that is used more than once. It is tested and reviewed.

---

## 2. A tour of this repository

This repository is itself laid out the way we recommend for a data science project. Let us look at the top
level.

In [ ]:
from pathlib import Path

REPO = Path("..").resolve()

for path in sorted(REPO.iterdir()):
    if path.name.startswith(".") or path.name in {"uv.lock", "__pycache__"}:
        continue
    kind = "dir " if path.is_dir() else "file"
    print(f"{kind}  {path.name}")

| Path | Purpose |
|---|---|
| `pyproject.toml` | The project's single configuration file: name, dependencies, tools. |
| `uv.lock` | Exact versions of every dependency, so everyone gets the same environment. |
| `src/churn/` | The reference package: the finished, tested version of the warm-up notebook. |
| `src/churn_capstone/` | Your package for the capstone (notebook 12). Almost empty on purpose. |
| `tests/` | Tests for `churn`. Run with `uv run pytest`. |
| `tests_capstone/` | Acceptance tests for your capstone package. |
| `configs/` | Experiment configurations as JSON files (read by `ExperimentConfig.from_json`). |
| `data/raw/` | The original data. Read only: never edit raw data by hand. |
| `notebooks/` | Exploration and teaching material. They import from `src/`. |
| `docs/` | Cheat sheet, further reading list and trainer guide. |
| `playground/` | Your scratch space. Ignored by git, except its README. |

Now the package itself. Each module has one job, which you can read from its name.

In [ ]:
for path in sorted((REPO / "src" / "churn").glob("*.py")):
    first_line = path.read_text().strip().splitlines()[0].strip('"')
    print(f"{path.name:<15} {first_line}")

### Why `src/`?

You could put the `churn/` folder directly in the repository root. The extra `src/` level looks like
bureaucracy, but it prevents a nasty class of bugs.

When you start Python, the current directory is put at the front of the import path. With a flat layout,
`import churn` might pick up the folder next to you instead of the installed package, depending on *where you
started Python from*. Your tests could then pass against code that is not what you ship.

With the `src` layout, the package is only importable because it is **installed**. Everybody (you, your
colleague, the test runner, the nightly job) imports exactly the same code.

## 3. What `uv sync` did for you

When you ran `uv sync` during setup, uv

1. created a virtual environment in `.venv/`,
2. installed the exact versions listed in `uv.lock`,
3. installed this project itself in **editable** mode: Python finds `churn` in `src/churn`, and changes you
   make to those files are visible immediately (restart the kernel to reload an already imported module).

That is why this notebook can import `churn` without any `sys.path` tricks.

In [ ]:
import churn

print("churn version:", churn.__version__)
print("imported from:", Path(churn.__file__).relative_to(REPO))

`__init__.py` decides what the package offers at the top level. Ours re-exports the most important classes, so
users can write `from churn import ChurnModel` instead of `from churn.model import ChurnModel`. The list
`__all__` documents this public API (and controls `from churn import *`).

In [ ]:
print(churn.__all__)

Everything is there. A complete run takes a handful of lines, because the package hides the details behind a
small public interface.

In [ ]:
from churn import ExperimentConfig, run_experiment

config = ExperimentConfig(data_path=REPO / "data" / "raw" / "telco_churn.csv", threshold=0.3)
model, result = run_experiment(config)

print(model)
print(result.summary())
print()
print(result.confusion)

> **Check your understanding**
>
> You open a notebook in `notebooks/` and run `import churn`. Later you edit `src/churn/evaluation.py` and run
> `import churn` again in the same notebook. Does the notebook see your change?
>
> <details><summary><b>Show answer</b></summary>
>
> No. Python imports a module only once per process; the second `import` reuses the module already in memory.
> The editable install means you do not need to *reinstall*, but you still need to **restart the kernel** (or use
> `importlib.reload`, which has its own pitfalls). In VS Code: "Restart" in the notebook toolbar.
>
> </details>

---

## 4. Modules, scripts and `if __name__ == "__main__":`

Every `.py` file is a **module**. Python sets a special variable `__name__` in each module:

- when the file is **imported**, `__name__` is the module's name (e.g. `"churn.data"`),
- when the file is **run as a program**, `__name__` is `"__main__"`.

The guard `if __name__ == "__main__":` therefore means "only run this when I am the program, not when somebody
imports me". Let us see it with a tiny module written to a temporary folder.

In [ ]:
import subprocess
import sys
import tempfile

module_code = """
def churn_rate(labels):
    return sum(labels) / len(labels)

print(f"__name__ is {__name__!r}")

if __name__ == "__main__":
    print("Running as a program: churn rate", churn_rate([1, 0, 0, 1]))
"""

with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / "metrics.py").write_text(module_code)

    print("--- python metrics.py ---")
    run = subprocess.run([sys.executable, "metrics.py"], cwd=tmp, capture_output=True, text=True)
    print(run.stdout)

    print("--- python -c 'import metrics' ---")
    run = subprocess.run(
        [sys.executable, "-c", "import metrics"], cwd=tmp, capture_output=True, text=True
    )
    print(run.stdout)

For a package, the program lives in `__main__.py`. Running `python -m churn` executes `src/churn/__main__.py`.
Open that file: it parses command line arguments, **configures logging** (the only place in the package that
does so, see notebook 09) and calls `run_experiment`.

`pyproject.toml` additionally registers a command called `churn-train` that points at the same `main()`
function:

```toml
[project.scripts]
churn-train = "churn.__main__:main"
```

In your terminal, from the repository root, both of these work:

```bash
uv run python -m churn --config configs/baseline.json
uv run churn-train --config configs/random_forest.json --save models/rf.joblib
```

Here is the first one, run from the notebook so you can see the output. Log messages go to stderr, the final
summary to stdout.

In [ ]:
run = subprocess.run(
    [sys.executable, "-m", "churn", "--config", "configs/baseline.json"],
    cwd=REPO,
    capture_output=True,
    text=True,
)
print("stderr (logs):")
print(run.stderr)
print("stdout (result):")
print(run.stdout)
print("exit code:", run.returncode)

The exit code matters: a scheduler or CI system only sees whether the program returned 0 (success) or
something else (failure). Our `main()` returns 1 when a `ChurnError` occurs, for example an invalid config.

## 5. `pyproject.toml`: one file for the whole project

`pyproject.toml` replaced the older `setup.py`, `setup.cfg` and `requirements.txt`. Let us read the one in this
repository with the standard library's TOML parser.

In [ ]:
import tomllib

with open(REPO / "pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

print("Top-level sections:", list(pyproject))
print("Tool sections:     ", list(pyproject["tool"]))
print()
print("requires-python:", pyproject["project"]["requires-python"])
print("dependencies:")
for dep in pyproject["project"]["dependencies"]:
    print("   ", dep)
print("packages built:", pyproject["tool"]["hatch"]["build"]["targets"]["wheel"]["packages"])

| Section | What it is for |
|---|---|
| `[project]` | Name, version, `requires-python`, runtime `dependencies`. |
| `[project.optional-dependencies]` | Extras you opt into, e.g. `uv sync --extra tracking` for DVC and MLflow. |
| `[project.scripts]` | Command line entry points such as `churn-train`. |
| `[dependency-groups]` | Tools for developers only (pytest, ruff, mypy). `uv sync` installs the `dev` group. |
| `[build-system]` | Which tool builds the package (here: hatchling). |
| `[tool.hatch...]` | Where the packages live (`src/churn`, `src/churn_capstone`). |
| `[tool.ruff]`, `[tool.pytest.ini_options]`, `[tool.mypy]` | Settings for each tool, so nobody has to remember command line flags. |

Dependencies are declared with **minimum** versions (`pandas>=2.2`), which keeps the project installable
alongside other packages. The **exact** versions everyone actually uses are pinned in `uv.lock`. To add a
dependency, do not edit the file by hand; run `uv add <package>` (or `uv add --dev <package>` for tools).

## 6. Quality tools you run from the terminal

```bash
uv run pytest                   # run the tests in tests/
uv run ruff check src tests     # lint: likely bugs and style problems
uv run ruff format src tests    # format the code consistently
uv run mypy                     # type-check src/churn (configured in pyproject.toml)
```

All four pass on `src/churn`. Make them pass on your capstone package too: it is the cheapest code review you
will ever get.

### Exercise 11.1
> **Core** · for everyone

Use the **public API** of the installed `churn` package (no copy-pasting of its code) to answer a question the
retention team asks: *how does the random forest configuration in `configs/random_forest.json` compare with the
default configuration?*

1. Load the random forest config with `ExperimentConfig.from_json`. The `data_path` inside it is relative to the
   repository root, so pass a loader explicitly: `CSVCustomerLoader(DATA_PATH)`.
2. Run `run_experiment` for both configs and store the two `ExperimentResult` objects in `default_result` and
   `forest_result`.
3. Print both summaries.

Use `help(run_experiment)` or `run_experiment?` if you are unsure about the arguments.

In [ ]:
# Exercise 11.1: your code here

from churn import CSVCustomerLoader, ExperimentConfig, run_experiment

DATA_PATH = REPO / "data" / "raw" / "telco_churn.csv"

default_result = None  # Your code here
forest_result = None  # Your code here

In [ ]:
# Check for exercise 11.1: run after solving it
from churn import ExperimentResult

assert isinstance(default_result, ExperimentResult), "default_result should be an ExperimentResult"
assert isinstance(forest_result, ExperimentResult), "forest_result should be an ExperimentResult"
assert default_result != forest_result, "the two runs should differ"
assert forest_result.roc_auc > 0.8
print(default_result.summary())
print(forest_result.summary())
print("All checks passed")

<details><summary><b>Show hint</b></summary>

`run_experiment` returns a tuple `(model, result)`. You only need the second element. The loader argument is
keyword `loader=`.

</details>

<details><summary><b>Show solution</b></summary>

```python
loader = CSVCustomerLoader(DATA_PATH)
forest_config = ExperimentConfig.from_json(REPO / "configs" / "random_forest.json")

_, default_result = run_experiment(ExperimentConfig(), loader=loader)
_, forest_result = run_experiment(forest_config, loader=loader)
```

Both configs use the same split (same `random_state` and `test_size`), so the comparison is fair. The forest
flags customers at threshold 0.3, the default config at 0.5, which explains most of the difference in recall.
Try `ExperimentConfig(threshold=0.3)` to compare like with like.

</details>

### Exercise 11.2
> **Stretch** · if you finish early

In your terminal (not in the notebook), from the repository root:

1. Run `uv run churn-train --config configs/random_forest.json --save models/rf.joblib -v`. What does `-v`
   change, and why is that controlled by the CLI and not by `churn.model`?
2. Create a config file `playground/bad.json` containing `{"test_size": 3}` and run the CLI with it. What exit
   code do you get (`echo $?` on Linux and macOS, `echo %errorlevel%` on Windows)? Where in the code is that
   decided?
3. Run `uv run python -c "import churn; print(churn.__file__)"` from inside the `playground/` folder. Why does
   it still work?

<details><summary><b>Show answers</b></summary>

1. `-v` sets the log level to DEBUG, so debug messages such as "Set TotalCharges=0 for 11 new customers" appear.
   The package only *emits* log records; the application (the CLI) decides what is shown and where.
2. Exit code 1. `ExperimentConfig.__post_init__` raises `ConfigError`; `main()` in `__main__.py` catches the
   `ChurnError` family at the boundary, logs it and returns 1.
3. The package is installed in the virtual environment (editable install), so the current directory does not
   matter.

</details>

### Exercise 11.3
> **Challenge** · for experienced programmers

The package has a single command. Design (on paper or in markdown) a second command `churn-score` that a
nightly job would run: it reads a saved model and a CSV of current customers *without* a `Churn` column, and
writes a CSV with `customerID` and churn probability, sorted from highest to lowest.

- Which existing functions and classes can you reuse unchanged?
- Which layer does the new code belong to, and which file would you put it in?
- What should happen if the file contains a customer with an unexpected `Contract` value? If a required column
  is missing?
- How would you test it without touching the real file system?

If you have time, implement it in `playground/` first, importing from `churn`.

<details><summary><b>Discussion points</b></summary>

- Reuse: `CSVCustomerLoader` (or `clean_customers`, which already accepts tables without `Churn`),
  `ChurnModel.load`, `ChurnModel.predict_proba`. No change to the domain code is needed: a good sign for the
  architecture.
- It is application-layer code: a second entry point next to `__main__.py` (e.g. `churn/score.py` with a
  `main()`), registered under `[project.scripts]`.
- Unknown categories: `OneHotEncoder` already encodes them as zeros and logs a warning. That is a business
  decision; you might instead want to fail. A missing column raises `SchemaError`, which the entry point should
  turn into exit code 1 so the scheduler alerts someone.
- Testing: write a tiny CSV into pytest's `tmp_path`, save a model fitted on a few rows there too, call
  `main([...])` with those paths and read the output CSV back.

</details>

---

## 7. Your package for the capstone

`src/churn_capstone/` already exists and is installed together with `churn`. Right now it only contains an
`__init__.py` with a docstring. In the capstone you will add modules to it (`data.py`, `errors.py`,
`model.py`, ...) so that the acceptance tests in `tests_capstone/` pass.

In [ ]:
import churn_capstone

capstone_dir = Path(churn_capstone.__file__).parent
print("Your package lives in:", capstone_dir.relative_to(REPO))
print("Modules so far:      ", sorted(p.name for p in capstone_dir.glob("*.py")))

---

## Summary

- Notebooks explore and explain; reusable logic lives in a package under `src/`, where it can be imported,
  tested and reviewed.
- The `src` layout plus an editable install (`uv sync`) means everyone imports the same code, from anywhere,
  without `sys.path` hacks. Restart the kernel to pick up changes.
- `__init__.py` defines the public API (re-exports and `__all__`); `__main__.py` and `[project.scripts]` turn
  the package into a command line program.
- `pyproject.toml` holds metadata, dependencies and tool settings; `uv.lock` pins exact versions.
- `pytest`, `ruff` and `mypy` run from the terminal and should all pass before you share code.

**Common pitfalls**

- Copying cells between notebooks instead of importing from the package.
- Editing package code and wondering why the notebook does not change (restart the kernel).
- Configuring logging or printing results inside library modules instead of in the entry point.
- Committing data, models or secrets (see the `.gitignore` of this repository and notebook X1).

---

## Sources and further reading

**Sources.** This notebook is based on:

- [The Python Tutorial, chapter 6: Modules](https://docs.python.org/3/tutorial/modules.html) and the docs on [`__main__`](https://docs.python.org/3/library/__main__.html).
- Python Packaging User Guide: [src layout vs flat layout](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/), [Writing your pyproject.toml](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/), [Packaging Python projects](https://packaging.python.org/en/latest/tutorials/packaging-projects/).
- Tool documentation: [uv projects](https://docs.astral.sh/uv/guides/projects/), [ruff](https://docs.astral.sh/ruff/), [mypy](https://mypy.readthedocs.io/en/stable/).

**For self-study.** Where to go next if you want to learn more:

- [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/) (website, beginner): a widely used project template with the reasoning behind each folder.
- [Made With ML](https://madewithml.com/) (free course, intermediate to advanced): from notebook to production ML, including packaging, testing and CI.

---

[← 10 · Testing with pytest](10_testing_with_pytest.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [12 · Capstone →](12_capstone.ipynb)